[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/kewel-corp/book-vol2/blob/main/notebooks/ch08_imbalance.ipynb)

# 第8章　クラス不均衡への対処：サンプリングと hard example mining

**『医療診断支援AI開発　第2巻　応用編 ― 本格実装から臨床導入まで（第2巻　応用編）』のコード**

本文に載っているコードを、章の順にそのまま収めています。紙面のコードは読んで理解するためのもの、こちらは動かすためのものです。

- Python 以外（シェル・YAML・Dockerfile など）は、実行環境が違うので**コードセルにせず、そのまま読める形で置いています**。使う場所を確かめてから実行してください。
- 抜粋である以上、上から順に実行するだけで通るとは限りません。データの取得先やパスは、お手元の環境に合わせてください。
- **教育・研究のためのコードです。患者データをこのノートブックに置かないでください。**

リポジトリ: https://github.com/kewel-corp/book-vol2

## 8.1　少数クラスの抽出頻度を増やす：オーバーサンプリング

In [ ]:
import torch
from torch.utils.data import DataLoader, WeightedRandomSampler
import numpy as np

labels = train_df["label"].values                    # 各症例のクラス
class_count = np.bincount(labels)
weight_per_class = 1.0 / class_count                 # 少ないクラスほど大きい重み
sample_weights = weight_per_class[labels]            # 各症例の重み
sampler = WeightedRandomSampler(sample_weights, num_samples=len(labels), replacement=True)

train_loader = DataLoader(train_ds, batch_size=16, sampler=sampler)  # shuffleの代わり

## 8.2　病変を含むパッチを、優先して切り出す

In [ ]:
from monai.transforms import RandCropByPosNegLabeld
crop = RandCropByPosNegLabeld(
    keys=["image", "label"], label_key="label",
    spatial_size=(96, 96, 96),
    pos=3, neg=1,                # 前景中心:背景中心の選択重み = 3:1（病変を含むパッチの比率ではない）
    num_samples=4,
)

## 8.3　モデルが誤判定しやすい症例を学習に多く用いる：hard example mining

In [ ]:
# バッチ内で、損失の大きい上位を選んで学習する（OHEM の簡易版）
losses = criterion_none(model(x), y)          # reduction="none" で各サンプルの損失
if losses.dim() > 1:                          # セグメンテーションでは画素ごとに出るので、
    losses = losses.flatten(1).mean(1)        # まず症例内で平均し、症例単位の1次元にする
k = max(1, int(len(losses) * 0.7))             # 上位70%。バッチサイズ1でも抽出数が0件にならないようにする
hard = losses.topk(k).values.mean()
optimizer.zero_grad(set_to_none=True)
hard.backward()
optimizer.step()

## 損失関数の側から不均衡に対処する：Tversky（トベルスキー）の β を調整する

In [ ]:
def tversky_loss(prob, target, beta=0.7, eps=1e-6):
    tp = (prob * target).sum()
    fp = (prob * (1 - target)).sum()
    fn = ((1 - prob) * target).sum()
    ti = (tp + eps) / (tp + (1 - beta) * fp + beta * fn + eps)
    return 1 - ti

## 判断がやさしい症例から学ばせる：カリキュラム学習

In [ ]:
# 難易度スコア（小さいほど判断がやさしい）で並べ、エポックとともに対象範囲を広げる
def curriculum_indices(difficulty, epoch, warmup=20):
    order = difficulty.argsort()                 # 判断がやさしい順に整列
    frac = min(1.0, 0.3 + 0.7 * epoch / warmup)  # 3割から徐々に全体へ
    k = int(len(order) * frac)
    return order[:k]                             # 今エポックで使う症例集合

## 長尾分布を扱う：サンプリング以外の対処

In [ ]:
def class_balanced_weights(class_count, beta=0.999):
    counts = np.asarray(class_count, dtype=float)
    if counts.ndim != 1 or not np.isfinite(counts).all():
        raise ValueError("class_count は有限値の1次元配列にする")
    if np.any(counts <= 0):
        # 学習 fold に1例もないクラスがある。eps で埋めても、学習例のない疾患をモデルが学べるようにはならない
        raise ValueError(f"症例数0のクラスがある: {np.where(counts <= 0)[0].tolist()}（分割か対象の設計を見直す）")
    if not (0 <= beta < 1):
        raise ValueError("beta は 0 以上 1 未満にする")
    eff = (1.0 - np.power(beta, counts)) / (1.0 - beta)  # 有効サンプル数
    w = 1.0 / eff
    return w / w.sum() * len(counts)          # 平均1に正規化して損失へ渡す

In [ ]:
counts_t = torch.as_tensor(class_count, dtype=torch.float32)
assert (counts_t > 0).all(), "症例数0のクラスがあると log(0) = -inf になる（上の関数と同じ入力の検査を通す）"
log_prior = torch.log(counts_t / counts_t.sum())
import torch.nn.functional as F

def logit_adjusted_ce(logits, y, tau=1.0):
    return F.cross_entropy(logits + tau * log_prior.to(logits.device), y)  # 推論時は補正前のlogits